# AIME² Minimal Experiment Notebook
This notebook generates synthetic data and reproduces the minimal validation for the unified AIME framework. Figures (PNG) and tables (TeX) are saved under `figs/` and `tables/` respectively.

In [1]:

import os, math, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "."
FIG_DIR = os.path.join(BASE, "figs")
TAB_DIR = os.path.join(BASE, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


In [2]:

def aime_unified(X, Y, W=None, Gamma=None):
    d, n = X.shape
    c, n2 = Y.shape
    assert n == n2
    if W is None:
        W = np.eye(n)
    if Gamma is None:
        Gamma = np.zeros((c, c))
    core = Y @ W @ Y.T + Gamma
    A = X @ W @ Y.T @ np.linalg.pinv(core)
    return A

def ridge_aime(X, Y, lam=1e-2, W=None):
    d, n = X.shape
    c, n2 = Y.shape
    if W is None:
        W = np.eye(n)
    Gamma = lam * np.eye(c)
    return aime_unified(X, Y, W=W, Gamma=Gamma)

def svd_aime(X, Y, r=5, W=None, Gamma=None):
    if W is None:
        W = np.eye(X.shape[1])
    d, n = X.shape
    c = Y.shape[0]
    if Gamma is None:
        Gamma = np.zeros((c, c))
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    r = min(r, U.shape[1])
    Ur = U[:, :r]
    Xr = Ur.T @ X
    core = Y @ W @ Y.T + Gamma
    A_svd = Ur @ (Xr @ W @ Y.T @ np.linalg.pinv(core))
    return A_svd

def fidelity(X, A, Y):
    num = np.linalg.norm(X - A @ Y, ord='fro')
    den = np.linalg.norm(X, ord='fro') + 1e-12
    return float(num / den)

def random_orthogonal(d, seed=None):
    rng = np.random.default_rng(seed)
    Q, _ = np.linalg.qr(rng.standard_normal((d, d)))
    if np.linalg.det(Q) < 0:
        Q[:, 0] *= -1.0
    return Q

def equivariance_error(X, Y, A_fn, **kwargs):
    d = X.shape[0]
    T = random_orthogonal(d)
    A1 = A_fn(T @ X, Y, **kwargs)
    A0 = A_fn(X, Y, **kwargs)
    num = np.linalg.norm(A1 - T @ A0, ord='fro')
    den = np.linalg.norm(A0, ord='fro') + 1e-12
    return float(num / den)

def stability_error(X, Y, A_fn, dX_scale=1e-2, dY_scale=1e-2, trials=5, **kwargs):
    errs = []
    for t in range(trials):
        dX = np.random.normal(scale=dX_scale, size=X.shape)
        dY = np.random.normal(scale=dY_scale, size=Y.shape)
        A0 = A_fn(X, Y, **kwargs)
        A1 = A_fn(X + dX, Y + dY, **kwargs)
        num = np.linalg.norm(A1 - A0, ord='fro')
        den = np.linalg.norm(dX, ord='fro') + np.linalg.norm(dY, ord='fro') + 1e-12
        errs.append(num / den)
    return float(np.mean(errs))

def cond_number_Y_block(Y, W=None, Gamma=None):
    if W is None:
        W = np.eye(Y.shape[1])
    if Gamma is None:
        Gamma = np.zeros((Y.shape[0], Y.shape[0]))
    M = Y @ W @ Y.T + Gamma
    s = np.linalg.svd(M, compute_uv=False)
    smax = np.max(s)
    smin = np.min(s) + 1e-15
    return float(smax / smin)

def make_block_correlated_X(d=20, n=200, rho=0.8, blocks=4, seed=42):
    rng = np.random.default_rng(seed)
    X = np.zeros((d, n))
    block_size = d // blocks
    for b in range(blocks):
        idx_start = b * block_size
        idx_end = (b+1) * block_size if b < blocks-1 else d
        k = idx_end - idx_start
        u = rng.standard_normal((1, n))
        noise = rng.standard_normal((k, n))
        X[idx_start:idx_end, :] = rho * u + math.sqrt(1 - rho**2) * noise
    return X

def make_linear_Y_from_X(X, c=3, noise_sigma=0.05, seed=123):
    rng = np.random.default_rng(seed)
    d, n = X.shape
    B = rng.standard_normal((c, d))
    Y = B @ X + rng.normal(scale=noise_sigma, size=(c, n))
    return Y, B


In [3]:

# Settings
d, n, c = 20, 200, 3
rho, blocks = 0.8, 4
noise_sigma = 0.05
svd_r = 5
ridge_lam = 1e-2
seed = 2025

# Data
X = make_block_correlated_X(d=d, n=n, rho=rho, blocks=blocks, seed=seed)
Y, B = make_linear_Y_from_X(X, c=c, noise_sigma=noise_sigma, seed=seed+1)
W = np.eye(n)

# Operators
A_aime = aime_unified(X, Y, W=W, Gamma=np.zeros((c, c)))
A_ridge = ridge_aime(X, Y, lam=ridge_lam, W=W)
A_svd  = svd_aime(X, Y, r=svd_r, W=W, Gamma=np.zeros((c, c)))

# Metrics
fid_aime = fidelity(X, A_aime, Y)
fid_ridge = fidelity(X, A_ridge, Y)
fid_svd = fidelity(X, A_svd, Y)

eq_aime = equivariance_error(X, Y, aime_unified, W=W, Gamma=np.zeros((c, c)))
eq_ridge = equivariance_error(X, Y, ridge_aime, W=W, lam=ridge_lam)
eq_svd = equivariance_error(X, Y, svd_aime, W=W, Gamma=np.zeros((c, c)), r=svd_r)

stab_aime = stability_error(X, Y, aime_unified, dX_scale=1e-2, dY_scale=1e-2, trials=10, W=W, Gamma=np.zeros((c, c)))
stab_ridge = stability_error(X, Y, ridge_aime, dX_scale=1e-2, dY_scale=1e-2, trials=10, W=W, lam=ridge_lam)
stab_svd = stability_error(X, Y, svd_aime, dX_scale=1e-2, dY_scale=1e-2, trials=10, W=W, Gamma=np.zeros((c, c)), r=svd_r)

# Store
labels = ["AIME", "Ridge-AIME", "SVD-AIME"]
metrics_df = pd.DataFrame({
    "Method": labels,
    "Fidelity": [fid_aime, fid_ridge, fid_svd],
    "Stability": [stab_aime, stab_ridge, stab_svd],
    "Equivariance": [eq_aime, eq_ridge, eq_svd],
})
metrics_df


/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/1607658847.py:97: RuntimeWarning: divide by zero encountered in matmul
  Y = B @ X + rng.normal(scale=noise_sigma, size=(c, n))
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/1607658847.py:97: RuntimeWarning: overflow encountered in matmul
  Y = B @ X + rng.normal(scale=noise_sigma, size=(c, n))
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/1607658847.py:97: RuntimeWarning: invalid value encountered in matmul
  Y = B @ X + rng.normal(scale=noise_sigma, size=(c, n))
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/1607658847.py:9: RuntimeWarning: divide by zero encountered in matmul
  core = Y @ W @ Y.T + Gamma
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/1607658847.py:9: RuntimeWarning: overflow encountered in matmul
  core = Y @ W @ Y.T + Gamma
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/1607658847.py:9: RuntimeWarning: invalid 

,Method,Fidelity,Stability,Equivariance
0,AIME,0.830392,0.002076,6.584313e-16
1,Ridge-AIME,0.830392,0.001968,7.535359e-16
2,SVD-AIME,0.840076,0.007751,7.763351e-15


In [4]:

# Figure 1: Fidelity
plt.figure()
plt.bar(range(3), [metrics_df['Fidelity'][i] for i in range(3)])
plt.xticks(range(3), metrics_df['Method'], rotation=0)
plt.ylabel("Normalized Reconstruction Error")
plt.title("Fidelity (lower is better)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_fidelity.png"), dpi=200)
plt.close()

# Stability vs lambda sweep
lam_grid = np.logspace(-6, 0, 15)
conds, stabs = [], []
for lam in lam_grid:
    conds.append(cond_number_Y_block(Y, W=W, Gamma=lam*np.eye(c)))
    stabs.append(stability_error(X, Y, ridge_aime, dX_scale=1e-2, dY_scale=1e-2, trials=3, W=W, lam=lam))

plt.figure()
plt.plot(lam_grid, stabs, marker='o')
plt.xscale('log')
plt.xlabel("Ridge $\lambda$")
plt.ylabel("Stability Error")
plt.title("Stability vs Regularization (Ridge-AIME)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_stability_vs_lambda.png"), dpi=200)
plt.close()

# Equivariance
plt.figure()
plt.bar(range(3), [metrics_df['Equivariance'][i] for i in range(3)])
plt.xticks(range(3), metrics_df['Method'], rotation=0)
plt.ylabel("Equivariance Error")
plt.title("Equivariance Check (lower is better)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_equivariance_error.png"), dpi=200)
plt.close()

# Condition number
plt.figure()
plt.plot(lam_grid, conds, marker='s')
plt.xscale('log'); plt.yscale('log')
plt.xlabel("Ridge $\lambda$")
plt.ylabel("cond$(YWY^T+\Gamma)$")
plt.title("Condition Number vs Regularization")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_condition_number_vs_lambda.png"), dpi=200)
plt.close()


<>:21: SyntaxWarning: invalid escape sequence '\l'
<>:42: SyntaxWarning: invalid escape sequence '\l'
<>:43: SyntaxWarning: invalid escape sequence '\G'
<>:21: SyntaxWarning: invalid escape sequence '\l'
<>:42: SyntaxWarning: invalid escape sequence '\l'
<>:43: SyntaxWarning: invalid escape sequence '\G'
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/3393885671.py:21: SyntaxWarning: invalid escape sequence '\l'
  plt.xlabel("Ridge $\lambda$")
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/3393885671.py:42: SyntaxWarning: invalid escape sequence '\l'
  plt.xlabel("Ridge $\lambda$")
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/3393885671.py:43: SyntaxWarning: invalid escape sequence '\G'
  plt.ylabel("cond$(YWY^T+\Gamma)$")
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/ipykernel_20244/1607658847.py:74: RuntimeWarning: divide by zero encountered in matmul
  M = Y @ W @ Y.T + Gamma
/var/folders/mk/k0r5f0ps1hb9kvffhhp7hxwc0000gn/T/i

In [6]:
mdf = metrics_df.copy()
mdf['Fidelity'] = mdf['Fidelity'].map(lambda x: f"{x:.4f}")
mdf['Stability'] = mdf['Stability'].map(lambda x: f"{x:.4f}")
mdf['Equivariance'] = mdf['Equivariance'].map(lambda x: f"{x:.4e}")

# --- metrics table ---
tex_metrics = r"""
\begin{table}[t]
\centering
\caption{Unified AIME minimal experiment: fidelity, stability, and equivariance (lower is better).}
\label{tab:aime_metrics}
\begin{tabular}{lccc}
\hline
Method & Fidelity & Stability & Equivariance \\
\hline
""".strip("\n")

for _, row in mdf.iterrows():
    tex_metrics += f"\n{row['Method']} & {row['Fidelity']} & {row['Stability']} & {row['Equivariance']} \\\\"

tex_metrics += r"""
\hline
\end{tabular}
\end{table}
"""

with open(os.path.join(TAB_DIR, "table_metrics.tex"), "w") as f:
    f.write(tex_metrics)


# --- settings table ---
settings = {
    "Feature dim $d$": d,
    "Samples $n$": n,
    "Output dim $c$": c,
    "Block correlation $\\rho$": rho,
    "Blocks": blocks,
    "Noise $\\sigma$": noise_sigma,
    "SVD rank $r$": svd_r,
    "Ridge $\\lambda$": ridge_lam
}

tex_settings = r"""
\begin{table}[t]
\centering
\caption{Experimental settings for the unified AIME minimal validation.}
\label{tab:aime_settings}
\begin{tabular}{lc}
\hline
Parameter & Value \\
\hline
""".strip("\n")

for k, v in settings.items():
    tex_settings += f"\n{k} & {v} \\\\"

tex_settings += r"""
\hline
\end{tabular}
\end{table}
"""

with open(os.path.join(TAB_DIR, "table_settings.tex"), "w") as f:
    f.write(tex_settings)

print("✅ TeX tables saved successfully to tables/")


✅ TeX tables saved successfully to tables/


In [8]:
!pip list > requirements.txt